# Numpy 之七：存储、读取与复制

---

前面都在内存里玩，这一篇讲**怎么把数组存到硬盘、再读回来**，
以及**复制数组时那个 `view` / `copy` 的坑**。

## 学习目标

1. 会存/读 **文本文件**（`savetxt` / `loadtxt`）—— 人眼可读，但慢
2. 会存/读 **二进制文件**（`save` / `load`）—— 快，但只能用 Numpy 读
3. 会一次存**多个数组**（`savez` / `savez_compressed`）
4. 彻底分清 **视图（view）** 和 **副本（copy）**


In [ ]:
import numpy as np
import os
import tempfile

# 【修正】原稿在这里直接用了一个没定义的 arr，单独运行会 NameError。
# 我们先把数据准备好，让这篇能独立跑通。
arr = np.array([[1., 2., 3.],
                [4., 5., 6.]])
print('要保存的数组：')
print(arr)
print('形状：', arr.shape, '，类型：', arr.dtype)

# 演示用的读写文件，统一丢到【系统临时目录】，不污染你的项目目录
_demo_dir = tempfile.mkdtemp(prefix='numpy_io_')
os.chdir(_demo_dir)
print('\n演示文件输出目录（系统临时目录）：')
print(' ', os.getcwd())


---
## 一、文本文件：`savetxt` / `loadtxt`

### 优点
- **人眼可读**，能用记事本打开看
- 跨语言通用（Excel、Matlab、R 都能读）

### 缺点
- 存的是字符串，**浮点会有精度损失**
- 读写**慢**，大数组非常明显

> 原稿这里 `np.savetxt('arr.txt', arr)` 就写完了，
> 但它前面没定义 `arr`，**单独打开这个 notebook 会直接报错**。现在补上了。


In [ ]:
np.savetxt('arr.txt', arr)          # 默认用空格分隔，科学计数法
print('已保存 arr.txt，内容如下：')
print('-' * 30)
with open('arr.txt', encoding='utf-8') as f:
    print(f.read())
print('-' * 30)


In [ ]:
# 读回来（读回来的 dtype 一定是【浮点】，因为文本没有类型信息）
arr_back = np.loadtxt('arr.txt')
print('读回来：')
print(arr_back)
print('dtype =', arr_back.dtype, '  <- 文本读回来一律是 float64')


### 常用参数：分隔符、格式、保留小数位


In [ ]:
np.savetxt('arr.csv', arr, delimiter=',', fmt='%.2f')
print('存成 CSV（逗号分隔，保留 2 位小数）：')
with open('arr.csv', encoding='utf-8') as f:
    print(f.read())

# 读 CSV 要指定 delimiter
print('用 delimiter="," 读回来：')
print(np.loadtxt('arr.csv', delimiter=','))


---
## 二、二进制文件：`save` / `load`

### 优点
- **快**，且**无损**（dtype、shape 全保留）
- 文件通常比文本小

### 缺点
- 记事本打开是乱码
- 只有 Numpy 能直接读

**文件后缀是 `.npy`。**


In [ ]:
np.save('arr.npy', arr)
size_txt = os.path.getsize('arr.txt')
size_npy = os.path.getsize('arr.npy')
print(f'arr.txt 大小：{size_txt} 字节')
print(f'arr.npy 大小：{size_npy} 字节')


In [ ]:
loaded = np.load('arr.npy')
print('读回来：')
print(loaded)
print('dtype =', loaded.dtype, '，shape =', loaded.shape, '  <- 类型和信息都完整保留 ✅')
print('和原数组完全一样吗？', np.array_equal(loaded, arr))


### 一次存多个数组：`savez`

`.npz` 就是一个"压缩包"，里面可以装很多个数组。
读回来得到的是一个**类似字典**的对象。


In [ ]:
a = np.arange(5)
b = np.linspace(0, 1, 5)
c = np.random.rand(2, 2)

np.savez('multi.npz', a=a, b=b, c=c)          # 不给名字的话，key 会是 arr_0, arr_1...
data = np.load('multi.npz')

print('npz 里装了什么：', list(data.keys()), '  <- 像字典的 key')
print('\na =', data['a'])
print('b =', data['b'])
print('c =')
print(data['c'])


### `savez` vs `savez_compressed`

| 函数 | 是否压缩 | 速度 | 文件大小 |
|---|---|---|---|
| `np.savez` | 不压缩 | 快 | 大 |
| `np.savez_compressed` | 压缩 | 慢 | 小 |

**数据量大、又不急着读**时用压缩版；**要求速度**就用普通版。


In [ ]:
np.savez_compressed('multi_z.npz', a=a, b=b, c=c)
print(f'multi.npz    大小：{os.path.getsize("multi.npz")} 字节')
print(f'multi_z.npz  大小：{os.path.getsize("multi_z.npz")} 字节')


### 三种存法怎么选？一张表说清

| 需求 | 用什么 | 后缀 |
|---|---|---|
| 想给人看 / 给 Excel 用 | `savetxt` + `delimiter=','` | `.csv` / `.txt` |
| 自己用、要快、要无损 | `save` / `load` | `.npy` |
| 一次存好几个数组 | `savez` / `savez_compressed` | `.npz` |

---

## 三、视图（view）与副本（copy）

这个知识点**必须掌握**，因为它会悄悄地"篡改"你的数据。


In [ ]:
arr1 = np.array([1, 2, 3, 4, 5])
arr2 = arr1.view()              # 视图：共享内存
print('arr1 =', arr1)
print('arr2 = arr1.view() =', arr2)

print('\narr2.base is arr1 吗？', arr2.base is arr1, '  <- True，说明 arr2 共享 arr1 的数据')

arr2[0] = 10                    # 改视图
print('\n改 arr2[0] = 10 之后：')
print('  arr2 =', arr2)
print('  arr1 =', arr1, '  <- 原数组跟着变了！')


**图示理解**：

```
view（视图）：              copy（副本）：
  arr1 ──┐                    arr1 ──> [1 2 3 4 5]
         ├──> [同一块内存]     arr2 ──> [1 2 3 4 5]   （各自独立）
  arr2 ──┘
  改一个，另一个跟着变        改一个，另一个不受影响
```

就像：
- **视图** = 给同一间房子开了扇新窗户，房里东西变了，哪个窗户看都一样
- **副本** = 照着原房子又盖了一间，怎么折腾都不影响原来那间


In [ ]:
arr3 = arr1.copy()              # 副本：独立内存
print('arr3 = arr1.copy() =', arr3)
print('arr3.base is arr1 吗？', arr3.base is arr1, '  <- False，独立')

arr3[0] = 100                   # 改副本
print('\n改 arr3[0] = 100 之后：')
print('  arr3 =', arr3)
print('  arr1 =', arr1, '  <- 原数组没变 ✅')


### 哪些操作产生视图，哪些产生副本？

| 操作 | 结果 | 说明 |
|---|---|---|
| `arr[1:4]`（切片） | **视图** | 最容易被忽略 |
| `arr.reshape()`（多数情况） | **视图** | 只改"看的方式"，不搬数据 |
| `arr.view()` | **视图** | 显式要视图 |
| `arr.copy()` | **副本** | 显式要副本 |
| `arr.astype(...)` | **副本** | 类型变了，必须新内存 |
| `arr[[0, 2]]`（花式索引） | **副本** | 取出的是不连续的新数据 |
| 布尔索引 `arr[arr > 3]` | **副本** | 同上 |
| `arr.flatten()` | **副本** | `ravel()` 才是视图 |

**记忆口诀：切片和 reshape 是"镜子"，其余大多是"复印件"。**


In [ ]:
# 验证几个
base = np.arange(6)
print('切片是视图吗？      ', base[1:3].base is not None)
print('reshape 是视图吗？  ', base.reshape(2, 3).base is base)
print('astype 是副本吗？   ', base.astype(float).base is None)
print('flatten 是副本吗？  ', base.reshape(2, 3).flatten().base is None)
print('ravel 是视图吗？    ', base.reshape(2, 3).ravel().base is not None)


---
## 四、清理演示文件

这篇为了演示产生了一堆 `.txt/.npy/.npz`。
我们已经把它们放在独立的临时目录里了，跑完可以顺手清掉：


In [ ]:
print('当前目录：', os.getcwd())
print('产生的文件：', sorted(os.listdir('.')))

# 想清理就取消下面两行的注释
# for f in os.listdir('.'):
#     os.remove(f)
# print('已清空')


---
## 五、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| 直接用未定义的变量存文件 | `NameError` | 先把数据准备好 |
| `loadtxt` 读 CSV 不写 `delimiter` | 报错 / 读成一行 | `np.loadtxt(f, delimiter=',')` |
| 以为文本能保留 dtype | 读回来全是 `float64` | 要无损用 `np.save` |
| 用 `arr.view()` 当复制 | 改一个另一个也变 | 要独立用 `.copy()` |
| 切片改了原数组 | 数据被"串改" | 切片是视图，改前先 `.copy()` |
| `np.load` 读 `.npz` 当数组用 | 是 `NpzFile` 不是 ndarray | 用 `data['key']` 取 |
| `savez` 不给 key | 名字变成 `arr_0` | 显式写 `a=a, b=b` |
| 把 `.npy` 当通用格式分享 | 别人打不开 | 给外部看用 CSV |
| `flatten` / `ravel` 搞混 | 一个改一个不改 | `flatten` 副本，`ravel` 视图 |

---

## 六、动手练习

**练习 1**：创建一个 `(3, 3)` 的随机数组，分别存成 `t.csv`（逗号分隔）和 `t.npy`，
比较两个文件大小，并说明为什么。

**练习 2**：把 `arr = np.arange(10)` 和 `arr2 = np.arange(10) * 10` 用 `savez` 存成一个文件，
再读出来打印。

**练习 3**：写代码验证"切片是视图"：改切片的元素，观察原数组是否变化。

**练习 4**：同一个数组，分别用 `view()` 和 `copy()` 得到两个新对象，
各改一个元素，说出哪个会影响原数组。

**练习 5**：把 `(2, 6)` 的数组存成 CSV，再读回来，
验证读回来的 `dtype` 和形状，并解释为什么 dtype 变了。

---

**答案思路（先自己做）**

```python
# 1
t = np.random.rand(3, 3)
np.savetxt('t.csv', t, delimiter=',')
np.save('t.npy', t)
# .npy 通常更小（二进制比文本紧凑），且读回来能保留 float64 精度

# 2
np.savez('ab.npz', arr=np.arange(10), arr2=np.arange(10) * 10)
d = np.load('ab.npz')
print(d['arr'], d['arr2'])

# 3
a = np.arange(5)
s = a[1:3]
s[0] = 999
print(a)      # 变了 -> 视图
```
